In [1]:
import sys
from pathlib import Path


PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == "dualtest_experiments" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "DUALTEST"))

from reference_model import ReferenceModel
from pipeline_functions import (
    load_membership_calibrator,
    make_groq_completion_fn,
    dualtest_pipeline_single,
)

c:\Users\isiva\miniconda3\envs\pytorch_m2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
import pandas as pd

DATASET_PATH = PROJECT_ROOT / "dataset" / "comparation_dataset" / "booktection_medium_eval_20books_10chunks.csv"

df_eval = pd.read_csv(DATASET_PATH)

def get_original_text(row):
    answer = str(row["answer"]).strip()
    return row[f"Example_{answer}"]

df_eval["original_text"] = df_eval.apply(get_original_text, axis=1)

df_eval.head()

,sample_id,book_id,label,length,answer,text,Example_A,Example_B,Example_C,Example_D,original_text
0,BT_0000,1984_-_George_Orwell,1,medium,A,Since he was arrested he had not been fed. He ...,Since he was arrested he had not been fed. He ...,He had not eaten anything since being arrested...,"Since his arrest, he had been given nothing to...",He had been given no food since being arrested...,Since he was arrested he had not been fed. He ...
1,BT_0001,The_Alchemist_-_Paulo_Coelho,1,medium,A,"“And I know the Soul of the World, because we ...","“And I know the Soul of the World, because we ...",The essence of the universe has confided in me...,I'm privy to the psyche of the cosmos from our...,The spirit of the world has confided in me dur...,"“And I know the Soul of the World, because we ..."
2,BT_0002,Oliver_Twist_-_Charles_Dickens,1,medium,A,"Master Bates followed, with a thoughtful count...","Master Bates followed, with a thoughtful count...","Master Bates trailed behind, looking pensive. ...","Master Bates came after, appearing thoughtful....","Master Bates followed behind, looking contempl...","Master Bates followed, with a thoughtful count..."
3,BT_0003,Unfortunately_Yours_-_Tessa_Bailey,0,medium,A,But August continues to try and fail and try a...,But August continues to try and fail and try a...,"However, August persists in attempting and fai...",But August keeps trying and failing and trying...,"However, August continues attempting and being...",But August continues to try and fail and try a...
4,BT_0004,The_Foxglove_King_-_Hannah_Whitten,0,medium,A,"It creaked, but if the sound woke anyone insid...","It creaked, but if the sound woke anyone insid...","The door made noise when opened, but no one in...","The door creaked open, but didn't seem to dist...",The door squeaked open but didn't appear to wa...,"It creaked, but if the sound woke anyone insid..."


In [3]:
model, threshold, threshold_info = load_membership_calibrator(
    PROJECT_ROOT / "results" / "membership_calibrator.joblib",
    PROJECT_ROOT / "results" / "membership_threshold.joblib",
)

In [4]:
reference = ReferenceModel(
    model_name="EleutherAI/pythia-410m",
    device="cpu",
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 292/292 [00:01<00:00, 151.95it/s]


In [10]:
#TARGET_MODEL = "llama-3.3-70b-versatile"
TARGET_MODEL = "llama-3.1-8b-instant"

completion_fn = make_groq_completion_fn(
    api_key="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",
    target_model=TARGET_MODEL,
)

In [11]:
out_path = PROJECT_ROOT / "results" / f"comparison_booktection_medium_{TARGET_MODEL.replace('-', '_').replace('.', '_')}.csv"

if out_path.exists():
    df_done = pd.read_csv(out_path)
    done_ids = set(df_done["sample_id"].astype(str))
    rows = df_done.to_dict("records")
    print("Retomando:", len(done_ids))
else:
    done_ids = set()
    rows = []

for _, row in df_eval.iterrows():
    sample_id = str(row["sample_id"])

    if sample_id in done_ids:
        continue

    try:
        result = dualtest_pipeline_single(
            source_text=row["original_text"],
            completion_fn=completion_fn,
            reference=reference,
            membership_model=model,
            threshold=threshold,
            target_model_name=TARGET_MODEL,
        )

        rows.append({
            "sample_id": row["sample_id"],
            "book_id": row["book_id"],
            "label": row["label"],
            "length": row["length"],
            "answer": row["answer"],
            **result,
        })

    except Exception as e:
        rows.append({
            "sample_id": row["sample_id"],
            "book_id": row.get("book_id"),
            "label": row.get("label"),
            "length": row.get("length"),
            "answer": row.get("answer"),
            "error": str(e),
            "target_model": TARGET_MODEL,
        })

    done_ids.add(sample_id)
    pd.DataFrame(rows).to_csv(out_path, index=False)

print("Guardado en:", out_path)

df_results = pd.DataFrame(rows)
df_results.head()

Guardado en: c:\Users\isiva\OneDrive\Escritorio\Ingenieria de software\NLP_Proyecto_Final\results\comparison_booktection_medium_llama_3_1_8b_instant.csv


,sample_id,book_id,label,length,answer,target_model,reference_model,run_length,edit_similarity,p_rlb,p_esb,log_p_esb,membership_probability,membership_threshold,suspicious,prefix,ground_truth,target_completion
0,BT_0000,1984_-_George_Orwell,1,medium,A,llama-3.1-8b-instant,EleutherAI/pythia-410m,3,0.288961,1.074971e-08,0.0,-142.800216,0.510111,0.750126,False,Since he was arrested he had not been fed. He ...,food was growing upon him. What he longed for ...,food was growing stronger by the minute. His s...
1,BT_0001,The_Alchemist_-_Paulo_Coelho,1,medium,A,llama-3.1-8b-instant,EleutherAI/pythia-410m,0,0.294304,1.000000e+00,0.0,-160.762492,0.569948,0.750126,False,"“And I know the Soul of the World, because we ...","no need for iron to be the same as copper, or ...","""...no separation between the earth, the trees..."
2,BT_0002,Oliver_Twist_-_Charles_Dickens,1,medium,A,llama-3.1-8b-instant,EleutherAI/pythia-410m,0,0.251515,1.000000e+00,0.0,-144.834191,0.472514,0.750126,False,"Master Bates followed, with a thoughtful count...",and a pewter pot on the trivet. There was a ra...,"hand, and a newspaper spread out before him. H..."
3,BT_0003,Unfortunately_Yours_-_Tessa_Bailey,0,medium,A,llama-3.1-8b-instant,EleutherAI/pythia-410m,0,0.239130,1.000000e+00,0.0,-191.794519,0.488275,0.750126,False,But August continues to try and fail and try a...,"gotten so lost in her speech, she didnt realiz...","shed a tear of pride as she watched him work, ..."
4,BT_0004,The_Foxglove_King_-_Hannah_Whitten,0,medium,A,llama-3.1-8b-instant,EleutherAI/pythia-410m,0,0.248428,1.000000e+00,0.0,-156.856366,0.475000,0.750126,False,"It creaked, but if the sound woke anyone insid...","shrugged. Not as such, no, but never say never...","just chuckled and shook his head. ""No, Val, we..."


In [12]:
df_ok = df_results.copy()

if "error" in df_ok.columns:
    df_ok = df_ok[df_ok["error"].isna()].copy()

df_ok[[
    "sample_id",
    "book_id",
    "label",
    "run_length",
    "edit_similarity",
    "log_p_esb",
    "membership_probability",
    "membership_threshold",
    "suspicious",
    "prefix",
    "ground_truth",
    "target_completion",
]].head()

,sample_id,book_id,label,run_length,edit_similarity,log_p_esb,membership_probability,membership_threshold,suspicious,prefix,ground_truth,target_completion
0,BT_0000,1984_-_George_Orwell,1,3,0.288961,-142.800216,0.510111,0.750126,False,Since he was arrested he had not been fed. He ...,food was growing upon him. What he longed for ...,food was growing stronger by the minute. His s...
1,BT_0001,The_Alchemist_-_Paulo_Coelho,1,0,0.294304,-160.762492,0.569948,0.750126,False,"“And I know the Soul of the World, because we ...","no need for iron to be the same as copper, or ...","""...no separation between the earth, the trees..."
2,BT_0002,Oliver_Twist_-_Charles_Dickens,1,0,0.251515,-144.834191,0.472514,0.750126,False,"Master Bates followed, with a thoughtful count...",and a pewter pot on the trivet. There was a ra...,"hand, and a newspaper spread out before him. H..."
3,BT_0003,Unfortunately_Yours_-_Tessa_Bailey,0,0,0.239130,-191.794519,0.488275,0.750126,False,But August continues to try and fail and try a...,"gotten so lost in her speech, she didnt realiz...","shed a tear of pride as she watched him work, ..."
4,BT_0004,The_Foxglove_King_-_Hannah_Whitten,0,0,0.248428,-156.856366,0.475000,0.750126,False,"It creaked, but if the sound woke anyone insid...","shrugged. Not as such, no, but never say never...","just chuckled and shook his head. ""No, Val, we..."
